In [ ]:
%pip install -q pandas numpy scikit-learn torch transformers sentence-transformers

# Toxic Comment Detection — Run Inference

**Pipeline stage**: `run-inference`  
**Upstream**: `select-model` → produces `select_model_output.json`  
**Downstream**: `assess-severity` / `recommend-moderation-action` → receives `run_inference_output.json`

This notebook loads the model selected by the upstream `select-model` stage, performs inference on a new comment, and writes a unified output payload into agent state for downstream business-facing skills.

### Supported model loading paths
- `tfidf_sklearn`: TF-IDF vectorizer + Logistic Regression / LinearSVC classifier
- `bert_embedding_lr`: frozen transformer encoder (e.g. ToxiGen-RoBERTa) + Logistic Regression classifier
- `sentence_transformer_finetuned`: fine-tuned MiniLM sequence-classification model


## 1. Setup Paths and Imports

In [ ]:
from __future__ import annotations

import json
import math
import pickle
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Any, Tuple

import numpy as np
import torch
from IPython.display import display
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

NOTEBOOK_DIR = Path.cwd().resolve()

def detect_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    markers = ["select_model_output.json", "models/selected_model_metadata.json", "raw_data"]
    for candidate in candidates:
        if sum((candidate / marker).exists() for marker in markers) >= 2:
            return candidate
    raise FileNotFoundError("Could not automatically detect project root from the current notebook location.")

PROJECT_ROOT = detect_project_root(NOTEBOOK_DIR)

SELECT_MODEL_OUTPUT_PATH = PROJECT_ROOT / "select_model_output.json"
TRAIN_METADATA_PATH = PROJECT_ROOT / "models" / "selected_model_metadata.json"
OUTPUT_PATH = PROJECT_ROOT / "run_inference_output.json"

# Replace this with the comment coming from your Gradio input / downstream pipeline state.
COMMENT_TEXT = "You are an absolute idiot and nobody wants you here."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 256

print(f"Project root              : {PROJECT_ROOT}")
print(f"Selected model state path : {SELECT_MODEL_OUTPUT_PATH}")
print(f"Train metadata path       : {TRAIN_METADATA_PATH}")
print(f"Inference output path     : {OUTPUT_PATH}")
print(f"Torch device              : {DEVICE}")

## 2. Helper Functions

In [ ]:
def load_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Required JSON not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def resolve_path(path_str: str) -> Path:
    path = Path(path_str)
    return path if path.is_absolute() else (PROJECT_ROOT / path).resolve()


def sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def probability_to_confidence(prob: float) -> float:
    return round(abs(prob - 0.5) * 2.0, 4)


def sanitize_text(text: str) -> str:
    if text is None:
        raise ValueError("COMMENT_TEXT cannot be None.")
    clean = str(text).strip()
    if not clean:
        raise ValueError("COMMENT_TEXT cannot be empty.")
    return clean


def mean_pool(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)


def get_artifact_config(selection_state: Dict[str, Any], train_meta: Dict[str, Any]) -> Dict[str, Any]:
    model_id = selection_state["selected_model_id"]
    artifact = dict(selection_state.get("artifact", {}))
    metadata_paths = train_meta.get("artifact_paths", {})

    default_configs = {
        "logistic_regression": {
            "type": "tfidf_sklearn",
            "vectorizer_path": metadata_paths.get("tfidf_vectorizer"),
            "model_path": metadata_paths.get("model_lr")
        },
        "lr": {
            "type": "tfidf_sklearn",
            "vectorizer_path": metadata_paths.get("tfidf_vectorizer"),
            "model_path": metadata_paths.get("model_lr")
        },
        "linear_svc": {
            "type": "tfidf_sklearn",
            "vectorizer_path": metadata_paths.get("tfidf_vectorizer"),
            "model_path": metadata_paths.get("model_linearsvc")
        },
        "linearsvc": {
            "type": "tfidf_sklearn",
            "vectorizer_path": metadata_paths.get("tfidf_vectorizer"),
            "model_path": metadata_paths.get("model_linearsvc")
        },
        "toxigen_bert_lr": {
            "type": "bert_embedding_lr",
            "base_model_name": train_meta.get("toxigen_model_name", "tomh/toxigen_roberta"),
            "model_path": metadata_paths.get("model_toxigen_lr")
        },
        "toxigen_lr": {
            "type": "bert_embedding_lr",
            "base_model_name": train_meta.get("toxigen_model_name", "tomh/toxigen_roberta"),
            "model_path": metadata_paths.get("model_toxigen_lr")
        },
        "minilm_ft": {
            "type": "sentence_transformer_finetuned",
            "model_path": metadata_paths.get("minilm_finetuned"),
            "base_model_name": train_meta.get("minilm_model_name", "sentence-transformers/all-MiniLM-L6-v2")
        }
    }

    merged = default_configs.get(model_id, {}).copy()
    merged.update({k: v for k, v in artifact.items() if v is not None})

    if "model_path" in merged and merged["model_path"]:
        merged["model_path"] = str(resolve_path(merged["model_path"]))
    if "vectorizer_path" in merged and merged.get("vectorizer_path"):
        merged["vectorizer_path"] = str(resolve_path(merged["vectorizer_path"]))

    return merged


def predict_with_tfidf_sklearn(text: str, artifact_cfg: Dict[str, Any]) -> Tuple[float, int, float]:
    vectorizer_path = Path(artifact_cfg["vectorizer_path"])
    model_path = Path(artifact_cfg["model_path"])

    with vectorizer_path.open("rb") as f:
        vectorizer = pickle.load(f)
    with model_path.open("rb") as f:
        model = pickle.load(f)

    features = vectorizer.transform([text])
    if hasattr(model, "predict_proba"):
        toxic_prob = float(model.predict_proba(features)[0, 1])
        raw_score = toxic_prob
    elif hasattr(model, "decision_function"):
        raw_score = float(model.decision_function(features)[0])
        toxic_prob = float(sigmoid(raw_score))
    else:
        toxic_label = int(model.predict(features)[0])
        toxic_prob = 1.0 if toxic_label == 1 else 0.0
        raw_score = toxic_prob

    predicted_label = int(toxic_prob >= 0.5)
    return toxic_prob, predicted_label, raw_score


def predict_with_embedding_lr(text: str, artifact_cfg: Dict[str, Any]) -> Tuple[float, int, float]:
    base_model_name = artifact_cfg["base_model_name"]
    model_path = Path(artifact_cfg["model_path"])

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    encoder = AutoModel.from_pretrained(base_model_name).to(DEVICE)
    encoder.eval()

    with model_path.open("rb") as f:
        classifier = pickle.load(f)

    enc = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        outputs = encoder(**enc)
        embeddings = mean_pool(outputs, enc["attention_mask"]).cpu().numpy()

    toxic_prob = float(classifier.predict_proba(embeddings)[0, 1])
    raw_score = toxic_prob
    predicted_label = int(toxic_prob >= 0.5)
    return toxic_prob, predicted_label, raw_score


def predict_with_finetuned_transformer(text: str, artifact_cfg: Dict[str, Any]) -> Tuple[float, int, float]:
    model_dir = artifact_cfg["model_path"]
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    model.eval()

    enc = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits.squeeze(0).detach().cpu()

    if logits.ndim == 0:
        raw_score = float(logits.item())
        toxic_prob = float(sigmoid(raw_score))
    elif logits.numel() == 1:
        raw_score = float(logits.item())
        toxic_prob = float(sigmoid(raw_score))
    else:
        probs = torch.softmax(logits, dim=-1).numpy()
        toxic_prob = float(probs[-1])
        raw_score = float(logits[-1].item())

    predicted_label = int(toxic_prob >= 0.5)
    return toxic_prob, predicted_label, raw_score


def run_inference(comment_text: str, selection_state: Dict[str, Any], train_meta: Dict[str, Any]) -> Dict[str, Any]:
    clean_text = sanitize_text(comment_text)
    artifact_cfg = get_artifact_config(selection_state, train_meta)
    model_type = artifact_cfg.get("type")
    threshold = float(selection_state.get("inference_threshold", train_meta.get("threshold", 0.5)))

    if model_type == "tfidf_sklearn":
        toxic_prob, _, raw_score = predict_with_tfidf_sklearn(clean_text, artifact_cfg)
    elif model_type == "bert_embedding_lr":
        toxic_prob, _, raw_score = predict_with_embedding_lr(clean_text, artifact_cfg)
    elif model_type == "sentence_transformer_finetuned":
        toxic_prob, _, raw_score = predict_with_finetuned_transformer(clean_text, artifact_cfg)
    else:
        raise ValueError(f"Unsupported artifact type: {model_type}")

    is_toxic = bool(toxic_prob >= threshold)
    predicted_label = "toxic" if is_toxic else "non-toxic"
    confidence = probability_to_confidence(toxic_prob)

    return {
        "comment_text": clean_text,
        "selected_model_id": selection_state["selected_model_id"],
        "selected_model_label": selection_state.get("selected_model_label"),
        "model_type": model_type,
        "threshold_used": threshold,
        "predicted_label": predicted_label,
        "predicted_class_id": int(is_toxic),
        "is_toxic": is_toxic,
        "toxicity_probability": round(float(toxic_prob), 4),
        "non_toxic_probability": round(float(1.0 - toxic_prob), 4),
        "confidence": confidence,
        "raw_score": round(float(raw_score), 4),
        "selection_justification": selection_state.get("selection_justification"),
        "bias_assessment": selection_state.get("bias_assessment"),
        "source_artifact": artifact_cfg,
        "inference_timestamp_utc": datetime.now(timezone.utc).isoformat()
    }


## 3. Load Agent State

In [ ]:
selection_state = load_json(SELECT_MODEL_OUTPUT_PATH)
train_meta = load_json(TRAIN_METADATA_PATH)
artifact_cfg = get_artifact_config(selection_state, train_meta)

print("Selected model ID    :", selection_state["selected_model_id"])
print("Selected model label :", selection_state.get("selected_model_label"))
print("Artifact config      :", json.dumps(artifact_cfg, indent=2))

## 4. Run Inference on New Comment

In [ ]:
inference_output = run_inference(COMMENT_TEXT, selection_state, train_meta)
display(inference_output)

## 5. Write Output to Agent State

In [ ]:
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(inference_output, f, indent=4, ensure_ascii=False)

print(f"Saved run inference output to: {OUTPUT_PATH}")

## 6. Downstream Output Contract

Downstream skills should read these fields from `run_inference_output.json`:

- `comment_text`: original user input after basic validation
- `selected_model_id`: internal model identifier
- `selected_model_label`: human-readable model name
- `model_type`: model loading strategy used for inference
- `threshold_used`: threshold applied to convert probability into the binary decision
- `predicted_label`: `toxic` or `non-toxic`
- `predicted_class_id`: `1` for toxic, `0` for non-toxic
- `is_toxic`: boolean version of the binary prediction
- `toxicity_probability`: probability assigned to the toxic class
- `non_toxic_probability`: complementary probability assigned to the clean class
- `confidence`: distance from decision boundary, scaled to `[0, 1]`
- `raw_score`: underlying decision score or toxic probability before downstream interpretation
- `selection_justification`: selection rationale propagated from upstream for traceability
- `bias_assessment`: fairness note propagated from upstream for transparency
- `source_artifact`: exact model and loader configuration used for this prediction
- `inference_timestamp_utc`: audit timestamp for this prediction run
